# Sonora TTS — Demo Server di Colab (GRATIS)

Jalankan cell 1–4 berurutan. Dapat URL publik → pasang ke Vercel env `TTS_URL` → redeploy Sonora → demo.
Catatan: runtime Colab mati kalau idle ±90 menit / tab ditutup. Untuk demo 1–2 jam aman.

In [ ]:
# 1) Install dependency (sekali per session)
!pip install -q fastapi "uvicorn[standard]" edge-tts pydantic

In [ ]:
# 2) Tulis server TTS
%%writefile server.py
import base64, os
import edge_tts
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title="Sonora TTS")
API_KEY = os.environ.get("SONORA_TTS_KEY", "")
VOICES = {"id": "id-ID-ArdiNeural", "en": "en-US-GuyNeural"}

class SpeakRequest(BaseModel):
    text: str = Field(min_length=1, max_length=2000)
    language: str = "id"
    voice: str = ""

@app.get("/health")
def health():
    return {"ok": True}

@app.post("/speak")
async def speak(req: SpeakRequest, x_sonora_key: str = Header(default="")):
    if API_KEY and x_sonora_key != API_KEY:
        raise HTTPException(status_code=401, detail="Unauthorized")
    voice = req.voice or VOICES.get(req.language, VOICES["id"])
    try:
        communicate = edge_tts.Communicate(req.text, voice)
        chunks = []
        async for chunk in communicate.stream():
            if chunk["type"] == "audio":
                chunks.append(chunk["data"])
        audio = b"".join(chunks)
    except Exception as e:
        raise HTTPException(status_code=502, detail=f"TTS failed: {e}")
    if not audio:
        raise HTTPException(status_code=502, detail="TTS returned empty audio")
    return {"audio": "data:audio/mpeg;base64," + base64.b64encode(audio).decode()}

In [ ]:
# 3) Jalankan server di background + tes lokal
import os
os.environ["SONORA_TTS_KEY"] = "demo-key-123"  # ganti sesukamu, samakan dengan TTS_KEY di Vercel
!nohup uvicorn server:app --host 127.0.0.1 --port 7860 > server.log 2>&1 &
import time; time.sleep(4)
!curl -s http://127.0.0.1:7860/health

In [ ]:
# 4) Expose ke internet + tampilkan URL publik otomatis
# Colab TIDAK punya IP publik, jadi kita pakai tunnel Cloudflare.
# Tunnel ini yang memberi kamu hostname publik (bukan IP Colab).
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import re, subprocess, threading, time
proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
public_url = None
def _watch():
    global public_url
    for line in proc.stdout:
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m and not public_url:
            public_url = m.group(0)
            print("\n✅ PUBLIC URL:", public_url)
            print("👉 TTS_URL =", public_url + "/speak")
threading.Thread(target=_watch, daemon=True).start()
time.sleep(15)
if not public_url:
    print("URL belum muncul — tunggu 10 detik lalu run ulang cell ini SAJA.")
else:
    import urllib.request
    print("Health check:", urllib.request.urlopen(public_url + "/health", timeout=30).read().decode())

## Setelah dapat URL

1. Copy URL `https://....trycloudflare.com` dari output cell 4.
2. Di Vercel → Project Sonora → Settings → Environment Variables:
   - `TTS_URL` = `https://....trycloudflare.com/speak`
   - `TTS_KEY` = `demo-key-123` (sama seperti di cell 3)
3. Redeploy Sonora.
4. Test: upload artwork → Generate → Play narration.

URL quick tunnel **berubah tiap Colab restart**. Untuk URL permanen gratis, pakai Render (lihat `tts-space/README.md`).